In [425]:
import pandas as pd
import os as os

%pip install lxml


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


01 — Extract: استخراج وتجميع البيانات

In [426]:
# Extract

html_files =os.listdir("C:\\Users\\hanan\\Downloads\\HTML_Files")
sales_dataframes=[]
for file_name in html_files:
    if file_name.lower().endswith(".html"):
        sales_df=pd.read_html(file_name)[0]

        sales_dataframes.append(sales_df)

# load data
combined_sales_df =pd.concat(sales_dataframes,ignore_index=True)

المرحلة 02 — Data Profiling

In [427]:
# Number of rows and columns
print("Dataset Shape:", combined_sales_df.shape)
# Column names
print("\nColumn Names:")
print(combined_sales_df.columns.tolist())
# Data types
print("\nData Types:")
print(combined_sales_df.dtypes)


Dataset Shape: (4862, 14)

Column Names:
['التاريخ', 'رقم الفاتورة', 'رقم البند', 'الفرع', 'رمز العميل', 'رمز المنتج', 'اسم المنتج', 'الفئة', 'الكمية', 'سعر الوحدة', 'الخصم %', 'إجمالي السطر', 'تكلفة الوحدة', 'حالة الطلب']

Data Types:
التاريخ             str
رقم الفاتورة        str
رقم البند           str
الفرع               str
رمز العميل          str
رمز المنتج          str
اسم المنتج          str
الفئة               str
الكمية            int64
سعر الوحدة       object
الخصم %           int64
إجمالي السطر    float64
تكلفة الوحدة    float64
حالة الطلب          str
dtype: object


In [428]:
# Missing values per column
def miising_values_per_columns(data):
      missing_values = data.isna().sum()
      # Missing values percentage
      missing_percentage =( data.isna().mean()*100).round(3)
      # Data quality summary
      missing_values_report = pd.DataFrame({
      "Missing Count":missing_values,
            "Missing %":missing_percentage
      })
      display(missing_values_report)
      
miising_values_per_columns(combined_sales_df)

,Missing Count,Missing %
التاريخ,0,0.000
رقم الفاتورة,0,0.000
رقم البند,0,0.000
الفرع,0,0.000
رمز العميل,5,0.103
رمز المنتج,0,0.000
اسم المنتج,0,0.000
الفئة,0,0.000
الكمية,0,0.000
سعر الوحدة,0,0.000


In [429]:
# Count fully duplicated rows
duplicate_rows_count = combined_sales_df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows_count)
# Display duplicated records for review
duplicate_records_df = combined_sales_df[
    combined_sales_df.duplicated(keep=False)
]

duplicate_records_df


Duplicate Rows: 0


,التاريخ,رقم الفاتورة,رقم البند,الفرع,رمز العميل,رمز المنتج,اسم المنتج,الفئة,الكمية,سعر الوحدة,الخصم %,إجمالي السطر,تكلفة الوحدة,حالة الطلب


03 — Transform: تنظيف البيانات وتحويلها

In [430]:
# 1. اكتشاف الصفوف التي تحتوي على تواريخ مختلفة
date_check = pd.to_datetime(
    combined_sales_df["التاريخ"],
    format="%d/%m/%Y",
    errors="coerce"
)
invalid_dates_df = combined_sales_df.loc[
    date_check.isna(),
    ["رقم البند", "التاريخ"]
]
print("عدد التواريخ مختلفة التنسيق:", len(invalid_dates_df))
invalid_dates_df


عدد التواريخ مختلفة التنسيق: 3


,رقم البند,التاريخ
972,س000986,2025-05-03
2438,س002486,2025-11-03
3920,س004137,2026-04-02


In [431]:
# 2. اكتشاف الصفوف التي تحتوي على أسعار غير قابلة للتحويل

# محاولة تحويل سعر الوحدة إلى رقم
price_check = pd.to_numeric(
    combined_sales_df["سعر الوحدة"],
    errors="coerce"
)

# استخراج الصفوف التي فشل تحويل سعرها
invalid_prices_df = combined_sales_df.loc[
    price_check.isna(),
    ["رقم البند", "سعر الوحدة"]
]

print("عدد الأسعار مختلفة التنسيق:", len(invalid_prices_df))
display(invalid_prices_df)


عدد الأسعار مختلفة التنسيق: 4


,رقم البند,سعر الوحدة
714,س000818,ر.س 115.00
1927,س002157,ر.س 140.00
3071,س003185,ر.س 112.20
4558,س004828,ر.س 127.50


In [432]:
profiling_sales_df = combined_sales_df.copy()

# إضافة نتائج اختبار التواريخ
profiling_sales_df["التاريخ"] = date_check

# إضافة نتائج اختبار الأسعار
profiling_sales_df["سعر الوحدة"] = price_check
      
miising_values_per_columns(profiling_sales_df)

,Missing Count,Missing %
التاريخ,3,0.062
رقم الفاتورة,0,0.000
رقم البند,0,0.000
الفرع,0,0.000
رمز العميل,5,0.103
رمز المنتج,0,0.000
اسم المنتج,0,0.000
الفئة,0,0.000
الكمية,0,0.000
سعر الوحدة,4,0.082


In [ ]:


combined_sales_df["سعر الوحدة"] = pd.to_numeric(combined_sales_df["سعر الوحدة"]    .astype(str)
    .str.replace("ر.س", "", regex=False)
    .str.strip(),errors="coerce")

combined_sales_df["التاريخ"] = pd.to_datetime(combined_sales_df["التاريخ"], 
    errors="coerce",
     format="mixed",
    dayfirst=True,)
combined_sales_df.dtypes

التاريخ         datetime64[us]
رقم الفاتورة               str
رقم البند                  str
الفرع                      str
رمز العميل                 str
رمز المنتج                 str
اسم المنتج                 str
الفئة                      str
الكمية                   int64
سعر الوحدة             float64
الخصم %                  int64
إجمالي السطر           float64
تكلفة الوحدة           float64
حالة الطلب                 str
dtype: object

In [440]:
miising_values_per_columns(combined_sales_df)

,Missing Count,Missing %
التاريخ,0,0.0
رقم الفاتورة,0,0.0
رقم البند,0,0.0
الفرع,0,0.0
رمز العميل,0,0.0
رمز المنتج,0,0.0
اسم المنتج,0,0.0
الفئة,0,0.0
الكمية,0,0.0
سعر بيع الوحدة,0,0.0


In [434]:
mis =combined_sales_df[combined_sales_df["رمز العميل"].isna()]


# الفواتير التي ينقصها رمز العميل
missing_invoices = combined_sales_df.loc[
    combined_sales_df["رمز العميل"].isna(),
    "رقم الفاتورة"
].unique()

# # عرض جميع بنود هذه الفواتير
invoice_check_df = combined_sales_df.loc[
    combined_sales_df["رقم الفاتورة"].isin(missing_invoices),
    ["رقم الفاتورة", "رقم البند", "رمز العميل"]
]



for i in mis.index:
    invoice_number = mis.loc[i,"رقم الفاتورة"]
  
    customer_ids = combined_sales_df.loc[
        (combined_sales_df["رقم الفاتورة"] == invoice_number)
        & (combined_sales_df["رمز العميل"].notna()),
        "رمز العميل"
    ].unique()
 
    combined_sales_df.loc[i,"رمز العميل"] =customer_ids[0]
combined_sales_df[combined_sales_df["رمز العميل"].isna()]


,التاريخ,رقم الفاتورة,رقم البند,الفرع,رمز العميل,رمز المنتج,اسم المنتج,الفئة,الكمية,سعر الوحدة,الخصم %,إجمالي السطر,تكلفة الوحدة,حالة الطلب


In [435]:

combined_sales_df = combined_sales_df.rename(
    columns={
        "إجمالي السطر": "صافي مبيعات المنتج",
        "سعر الوحدة": "سعر بيع الوحدة"
    }
)
combined_sales_df

,التاريخ,رقم الفاتورة,رقم البند,الفرع,رمز العميل,رمز المنتج,اسم المنتج,الفئة,الكمية,سعر بيع الوحدة,الخصم %,صافي مبيعات المنتج,تكلفة الوحدة,حالة الطلب
0,2025-01-01,ط00115,س000209,جدة - الروضة,ع0285,م104,إكسسوار شعر,إكسسوارات,1,30.0,5,28.50,15.99,مكتمل
1,2025-01-01,ط00115,س000210,جدة - الروضة,ع0285,م004,فستان سهرة,نسائي,1,270.0,0,270.00,108.62,مكتمل
2,2025-01-01,ط00119,س000214,الرياض - العليا,ع0015,م058,ثوب رجالي,رجالي,2,210.0,0,420.00,75.07,مكتمل
3,2025-01-01,ط00119,س000215,الرياض - العليا,ع0015,م030,قميص كتان,نسائي,2,125.0,0,250.00,52.96,مكتمل
4,2025-01-01,ط00013,س000024,الرياض - العليا,ع0067,م009,عباءة كلاسيكية,نسائي,1,230.0,0,230.00,96.87,مكتمل
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4857,2026-06-30,ط02744,س004852,الرياض - النخيل,ع0058,م001,فستان يومي,نسائي,1,117.3,0,117.30,56.05,مكتمل
4858,2026-06-30,ط02744,س004853,الرياض - النخيل,ع0058,م058,ثوب رجالي,رجالي,1,214.2,0,214.20,78.07,مكتمل
4859,2026-06-30,ط02638,س004659,الرياض - العليا,ع0319,م055,هودي,رجالي,2,142.8,0,285.60,70.05,مكتمل
4860,2026-06-30,ط02608,س004610,جدة - الروضة,ع0747,م051,بنطال قماش,رجالي,1,193.8,5,184.11,94.03,مكتمل


In [436]:

# Calculate expected net sales
expected_sales = (
    combined_sales_df["الكمية"]
    * combined_sales_df["سعر بيع الوحدة"]
    * (1 - combined_sales_df["الخصم %"] / 100)
).round(2)

# Compare expected and recorded sales
sales_difference = (
    expected_sales
    - combined_sales_df["صافي مبيعات المنتج"]
).abs().round(2)
# Find records with calculation errors
incorrect_sales_df = combined_sales_df.loc[
    (sales_difference > 0.01)
    | expected_sales.isna()
    | combined_sales_df["صافي مبيعات المنتج"].isna()
]

print("عدد السجلات غير المتطابقة:", len(incorrect_sales_df))
incorrect_sales_df[
        [
            "رقم البند",
            "الكمية",
            "سعر بيع الوحدة",
            "الخصم %",
            "صافي مبيعات المنتج"
        ]
]

عدد السجلات غير المتطابقة: 0


,رقم البند,الكمية,سعر بيع الوحدة,الخصم %,صافي مبيعات المنتج


In [437]:
# 1. الكميات الصفرية أو السالبة

invalid_quantity =(combined_sales_df["الكمية"] <= 0).sum()
# 2. الأسعار الصفرية أو السالبة

invalid_price = (combined_sales_df["سعر بيع الوحدة"] <0).sum()
# 3. الخصومات السالبة

negative_discount=     (combined_sales_df["الخصم %"] <0).sum()
# 4. الخصومات الأكبر من 100%

high_discount =     (combined_sales_df["الخصم %"] >100).sum()

# عدد أرقام البنود المكررة
duplicate_item_count = (
    combined_sales_df["رقم البند"].duplicated().sum()
)
duplicate_item_count
# Create Validation Report
validation_report = pd.DataFrame([{
    "Invalid Quantity": invalid_quantity,
    "Invalid Price": invalid_price,
    "Negative Discount": negative_discount,
    "Discount Above 100%": high_discount,
    "duplicate_item_count":duplicate_item_count
}])
validation_report

,Invalid Quantity,Invalid Price,Negative Discount,Discount Above 100%,duplicate_item_count
0,0,0,0,0,0


In [438]:
# عرض جميع السجلات التي تشترك في رقم بند
duplicate_items_df = combined_sales_df.loc[
    combined_sales_df["رقم البند"].duplicated(keep=False),
    ["رقم الفاتورة", "رقم البند", "اسم المنتج", "الفرع"]
]

duplicate_items_df


,رقم الفاتورة,رقم البند,اسم المنتج,الفرع


04 — Load: تحميل البيانات

In [439]:

combined_sales_df.to_excel("data ETL.xlsx", index=False)

